# 99 — Compare: pgvector vs Qdrant vs Weaviate vs Milvus vs Chroma vs LanceDB

Every DB notebook (`01`–`06`) loads the same Word/PDF documents with **LangChain** (`PyPDFLoader` + `Docx2txtLoader`),
runs the same ingestion gates and section-aware chunking, uses the same Pydantic schema and the same embedding model,
then writes its metrics to `data/results/<db>.json`. This notebook only **reads those files** — it does not touch any database.

| # | Section |
|---|---|
| 0 | Setup + load every `data/results/*.json` |
| 1 | Comparison table (executed, chunks, quarantined, hit@1, hit@5, p50 latency, isolation) |
| 2 | Feature matrix per DB (index layout, tenant isolation, ACL filter, alias, embedded vs server, system of record) |
| 3 | DBs not executed + what unblocks them |
| 4 | Sanity checks + takeaways |

## 0. Setup + load results

In [ ]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "data" / "results"
ORDER = ["pgvector", "qdrant", "weaviate", "milvus", "chroma", "lancedb"]
NOTEBOOK = {"pgvector": "01_pgvector.ipynb", "qdrant": "02_qdrant.ipynb", "weaviate": "03_weaviate.ipynb",
            "milvus": "04_milvus.ipynb", "chroma": "05_chroma.ipynb", "lancedb": "06_lancedb.ipynb"}

raw = {}
for p in sorted(RESULTS.glob("*.json")):
    try:
        d = json.loads(p.read_text(encoding="utf-8"))
    except json.JSONDecodeError as e:
        print(f"skip {p.name}: {e}")
        continue
    raw[d.get("db", p.stem)] = d
print("results found:", sorted(raw))
assert raw, f"no results in {RESULTS} - run the DB notebooks first"

def is_executed(d):
    # pgvector's file predates the 'executed' key: treat it as executed if it carries real metrics
    if "executed" in d:
        return bool(d["executed"])
    return d.get("chunks_total") is not None and d.get("hit@5") is not None

## 1. Comparison table

In [ ]:
rows = []
for db in ORDER + sorted(set(raw) - set(ORDER)):
    d = raw.get(db)
    if d is None:
        rows.append({"db": db, "executed": False, "chunks_total": None, "quarantined": None, "hit@1": None,
                     "hit@5": None, "latency_p50_ms": None, "isolation": "(no results file)",
                     "notebook": NOTEBOOK.get(db, "")})
        continue
    rows.append({"db": db, "executed": is_executed(d), "chunks_total": d.get("chunks_total"),
                 "quarantined": d.get("quarantined"), "hit@1": d.get("hit@1"), "hit@5": d.get("hit@5"),
                 "latency_p50_ms": d.get("latency_p50_ms"), "isolation": d.get("isolation", ""),
                 "notebook": NOTEBOOK.get(db, "")})
cmp_df = pd.DataFrame(rows).set_index("db")
for c in ["chunks_total", "quarantined"]:
    cmp_df[c] = pd.array(cmp_df[c], dtype="Int64")
for c in ["hit@1", "hit@5", "latency_p50_ms"]:
    cmp_df[c] = pd.to_numeric(cmp_df[c], errors="coerce")
cmp_df

Same table, styled: a bar on p50 latency (shorter is faster) and a highlight on the fastest executed DB. `Styler.bar` needs only jinja2, not matplotlib. Latency is measured on one laptop with ~28 chunks per industry, so treat it as a rough signal, not a benchmark.

In [ ]:
view = cmp_df.drop(columns=["isolation"])
try:
    styled = (view.style
              .bar(subset=["latency_p50_ms"], color="#9ecae1", vmin=0)
              .highlight_min(subset=["latency_p50_ms"], color="#c7e9c0")
              .format({"hit@1": "{:.2f}", "hit@5": "{:.2f}", "latency_p50_ms": "{:.2f}"}, na_rep="-"))
    display(styled)
except Exception as e:                      # styling is optional; the plain table above is the source of truth
    print("styling unavailable:", type(e).__name__, e)
    display(view)

In [ ]:
ex = cmp_df[cmp_df["executed"]].sort_values("latency_p50_ms")
print("Executed DBs ranked by p50 latency (ms):")
for db, r in ex.iterrows():
    print(f"  {db:<9} {r['latency_p50_ms']:>6.2f}  hit@1={r['hit@1']:.2f} hit@5={r['hit@5']:.2f} chunks={r['chunks_total']}")

## 2. Feature matrix

What each notebook actually builds. Index naming is the same everywhere: one index per industry named
`{industry}_v1_bgesmall_384` (BAAI/bge-small-en-v1.5, 384 dims, cosine), plus an alias `{industry}` used as the
blue/green switch point when the DB has one. "Isolation" names what really stops one tenant from reading another's data.

In [ ]:
FEATURES = {
    "pgvector": {
        "industry index": "table per industry, HNSW (vector_cosine_ops)",
        "tenant isolation": "Postgres Row-Level Security on tenant_id (enforced by the server for the app role)",
        "ACL filter": "allowed_groups && ARRAY[...] + status/validity in the SQL WHERE (pre-filter)",
        "alias": "yes - view {industry} over the versioned table",
        "runs as": "server (Postgres container)",
        "system of record": "Postgres sources table (same DB)",
    },
    "qdrant": {
        "industry index": "collection per industry, HNSW cosine, typed payload indexes",
        "tenant isolation": "mandatory tenant filter in safe_search + tenant_id keyword index is_tenant=True (co-location only, no RLS)",
        "ACL filter": "MatchAny(allowed_groups) + status + validity in query_points filter (pre-filter)",
        "alias": "yes - native collection alias",
        "runs as": "server (Qdrant container)",
        "system of record": "SQLite data/sor/qdrant.db",
    },
    "weaviate": {
        "industry index": "collection per industry, HNSW cosine, self-provided vectors",
        "tenant isolation": "native multi-tenancy: one shard per tenant_id, server rejects tenant-less queries (+ tenant_id filter)",
        "ACL filter": "contains_any(allowed_groups) + status + validity in near_vector filter (pre-filter)",
        "alias": "yes - collection alias (Weaviate >= 1.32), dict fallback",
        "runs as": "server (Weaviate container)",
        "system of record": "SQLite data/sor/weaviate.db",
    },
    "milvus": {
        "industry index": "collection per industry, HNSW (M=16, efC=200) COSINE, explicit schema",
        "tenant isolation": "tenant_id partition key + mandatory tenant filter (partition key alone is not a boundary)",
        "ACL filter": "ARRAY_CONTAINS_ANY(allowed_groups) + status + validity in search filter (pre-filter)",
        "alias": "yes - native collection alias",
        "runs as": "server (Milvus standalone container)",
        "system of record": "SQLite data/sor/milvus.db",
    },
    "chroma": {
        "industry index": "collection per industry, HNSW cosine space",
        "tenant isolation": "filter-only: tenant_id in the where clause; no auth/RLS (embedded)",
        "ACL filter": "$contains on allowed_groups ($or for several) + status + int-date validity in where (pre-filter)",
        "alias": "no - CURRENT dict (documented)",
        "runs as": "embedded (PersistentClient, data/chroma)",
        "system of record": "SQLite data/sor/chroma.db",
    },
    "lancedb": {
        "industry index": "table per industry, exact KNN (no ANN index at this size), PyArrow schema from Pydantic",
        "tenant isolation": "filter-only: tenant_id in the where pre-filter; no auth/RLS (embedded)",
        "ACL filter": "array_has_any(allowed_groups) + status + validity in where (pre-filter, FilterExec below KNN)",
        "alias": "no - CURRENT dict + SQLite aliases table",
        "runs as": "embedded (lancedb.connect, data/lancedb)",
        "system of record": "SQLite data/sor/lancedb.db",
    },
}
feat_df = pd.DataFrame.from_dict(FEATURES, orient="index")
feat_df.insert(0, "executed", [bool(cmp_df.loc[db, "executed"]) if db in cmp_df.index else False for db in feat_df.index])
with pd.option_context("display.max_colwidth", None):
    display(feat_df)

Isolation statement each notebook wrote into its own results file (the notebook's own words):

In [ ]:
for db, r in cmp_df.iterrows():
    print(f"- {db}: {r['isolation']}")

In [ ]:
# Same matrix as markdown, for pasting into docs
cols = list(feat_df.columns)
lines = ["| db | " + " | ".join(cols) + " |", "|---" * (len(cols) + 1) + "|"]
for db, r in feat_df.iterrows():
    lines.append(f"| {db} | " + " | ".join(str(r[c]) for c in cols) + " |")
from IPython.display import Markdown
display(Markdown("\n".join(lines)))

## 3. DBs not executed

In [ ]:
BLOCKERS = {
    "weaviate": "Docker Hub rate limit on the image pull. Run `podman login docker.io`, then "
                "`podman compose --profile weaviate up -d`, then execute notebooks/03_weaviate.ipynb (twice).",
    "milvus":   "Milvus images also come from Docker Hub: `podman login docker.io`, then "
                "`podman compose --profile milvus up -d`, then execute notebooks/04_milvus.ipynb.",
}
not_run = cmp_df[~cmp_df["executed"]]
if not_run.empty:
    print("All DBs executed.")
for db, r in not_run.iterrows():
    note = raw.get(db, {}).get("note") or BLOCKERS.get(db, "no results yet - execute its notebook")
    print(f"NOT EXECUTED: {db} ({r['notebook']})\n  why/next: {note}\n  its metrics in section 1 are empty (not zero).")

## 4. Sanity checks + takeaways

In [ ]:
ex = cmp_df[cmp_df["executed"]]
assert len(ex) >= 1, "at least one DB must have executed"
# Every executed DB ran the same corpus through the same gates, so these must agree
assert ex["chunks_total"].nunique() == 1, f"chunk counts differ: {ex['chunks_total'].to_dict()}"
assert ex["quarantined"].nunique() == 1, f"quarantine counts differ: {ex['quarantined'].to_dict()}"
assert (ex["hit@5"] >= ex["hit@1"]).all(), "hit@5 must be >= hit@1"
print("executed:", list(ex.index))
print("chunks_total (all equal):", int(ex["chunks_total"].iloc[0]), "| quarantined:", int(ex["quarantined"].iloc[0]))
print("hit@1 range:", ex["hit@1"].min(), "-", ex["hit@1"].max(), "| hit@5 range:", ex["hit@5"].min(), "-", ex["hit@5"].max())
print("fastest p50:", ex["latency_p50_ms"].idxmin(), f"{ex['latency_p50_ms'].min():.2f} ms",
      "| slowest p50:", ex["latency_p50_ms"].idxmax(), f"{ex['latency_p50_ms'].max():.2f} ms")
print("not executed:", list(cmp_df.index[~cmp_df["executed"]]) or "none")

**Takeaways**

- **Retrieval quality is the same everywhere.** Same chunks, same embeddings, same cosine metric, so hit@k is
  decided by the pipeline, not by the DB. Choose the DB for its operations and isolation, not for recall at this size.
- **Isolation is where the DBs really differ.** Only pgvector (RLS) and Weaviate (native multi-tenancy, once it runs)
  enforce tenant separation inside the server. Qdrant and Milvus co-locate tenants (is_tenant / partition key) but still
  rely on the mandatory filter in `safe_search`. Chroma and LanceDB are embedded and filter-only: anyone with the files
  can read every tenant.
- **Alias / blue-green.** pgvector, Qdrant, Milvus and Weaviate have a server-side pointer. Chroma and LanceDB keep a
  `CURRENT` dict, so the switch point lives in app code.
- **Latency** is a few milliseconds for all of them at ~224 chunks; the differences shown here will not survive a
  realistic data volume, so benchmark at your own scale before deciding.